# 06 · Selección del docente

Promueve al de mejor macro-F1 en `test`, sin mirar costo, y lo deja en `teacher.json` para la destilación. Macro-F1 y no accuracy porque el corpus está desbalanceado.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/CCSandoval/ButterflyModeling.git"

if Path("/kaggle").exists():
    CLON = "/tmp/ButterflyModeling"
    if not Path(CLON).exists():                     # el repo es público: clonar no pide credenciales
        subprocess.run(["git", "clone", "--depth", "1", REPO, CLON], check=True)
    sys.path.insert(0, CLON)
    os.chdir(CLON)
    print("clon en", CLON, "·", subprocess.run(
        ["git", "log", "-1", "--format=%h %s"], cwd=CLON,
        capture_output=True, text=True).stdout.strip())

    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        print("sin GITHUB_TOKEN: no se publicará")

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "corpus.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Raíz del proyecto:", ROOT)

In [ ]:
import pandas as pd

from scripts import corpus, evalStats
from scripts import modelRegistry as registro
from scripts.vizStyle import SERIE_1, SERIE_2, applyStyle

CORPUS = corpus.manifiesto()["creado"]

# clave (arquitectura, descongelado): congelado y fine-tuning son regímenes
# distintos y ambos deben verse en la tabla, no solo el más reciente
runPorArquitectura = {}
for run in registro.listRuns():
    clave = (run.get("architecture"), run.get("finetuning", 0.0))
    if (run.get("role") != "destilado" and run.get("dataset") == CORPUS
            and clave not in runPorArquitectura):
        runPorArquitectura[clave] = run



## 1 · Calidad y costo de cada candidato

In [ ]:
filas = []
for (arquitectura, descongelado), run in runPorArquitectura.items():
    ruta = registro.rutaModelo(run["run_id"])
    modelo = registro.modeloInferencia(run["run_id"])
    latencia, desviacion = evalStats.medirLatencia(modelo)
    filas.append({
        "arquitectura": arquitectura,
        "descongelado": descongelado,
        "test_accuracy": run["test_accuracy"],
        "test_macro_f1": run["test_macro_f1"],
        "validate_macro_f1": run["validate_macro_f1"],
        "parametros_M": round(modelo.count_params() / 1e6, 2),
        "tamano_MB": round(ruta.stat().st_size / 1e6, 1),
        "latencia_ms": round(latencia, 1),
        "latencia_sd_ms": round(desviacion, 1),
        "minutos_entrenamiento": run["training_minutes"],
        "run_id": run["run_id"],
    })

comparacion = pd.DataFrame(filas).sort_values("validate_macro_f1", ascending=False).reset_index(drop=True)
comparacion

## 2 · Accuracy vs macro-F1

In [ ]:
applyStyle()
import matplotlib.pyplot as plt

orden = comparacion.sort_values("validate_macro_f1")
y = range(len(orden))

fig, ax = plt.subplots(figsize=(8, 0.6 * len(orden) + 1.5))
ax.barh([i - 0.18 for i in y], orden["test_accuracy"], height=0.32, color=SERIE_1, label="accuracy")
ax.barh([i + 0.18 for i in y], orden["test_macro_f1"], height=0.32, color=SERIE_2, label="macro F1")
ax.set_yticks(list(y), orden["arquitectura"])
ax.set(xlim=(0, 1), xlabel="Test", title="Candidatos a docente")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## 3 · Docente elegido

Se promueve, se copia a `outputs/model.keras` y se deja en `teacher.json`
para que las notebooks de destilación lo lean sin elegirlo a mano.

In [ ]:
mejor = comparacion.iloc[0]
run = runPorArquitectura[(mejor["arquitectura"], mejor["descongelado"])]

registro.writeJson(registro.OUTPUTS_DIR / "teacher.json", {
    "run_id": run["run_id"],
    "architecture": mejor["arquitectura"],
    "dataset": CORPUS,
    "test_accuracy": float(mejor["test_accuracy"]),
    "test_macro_f1": float(mejor["test_macro_f1"]),
    "validate_macro_f1": float(mejor["validate_macro_f1"]),
    "latencia_ms": float(mejor["latencia_ms"]),
    "latencia_sd_ms": float(mejor["latencia_sd_ms"]),
    "parametros_M": float(mejor["parametros_M"]),
})

segundo = comparacion.iloc[1]
print(f"Docente: {mejor['arquitectura']} · macro-F1 {mejor['test_macro_f1']:.4f}")
print(f"Ventaja sobre el siguiente ({segundo['arquitectura']}): "
      f"{mejor['test_macro_f1'] - segundo['test_macro_f1']:+.4f}")

estudiante = registro.readJson(registro.OUTPUTS_DIR / "student.json")
if estudiante:
    brecha = mejor["test_macro_f1"] - estudiante["test_macro_f1"]
    if estudiante["run_id"] == run["run_id"]:
        print(f"\nel mismo run gana ambos roles: destilar no aplica")
    elif brecha <= 0:
        print(f"\nel estudiante {estudiante['architecture']} supera a este docente por "
              f"{-brecha:+.4f}: destilar no tiene margen")
    else:
        print(f"\nbrecha con el estudiante {estudiante['architecture']}: {brecha:+.4f}")

from pathlib import Path

from scripts.kaggle import publicarArchivos
if Path("/kaggle").exists() and os.environ.get("GITHUB_TOKEN"):
    publicarArchivos(f"Select {mejor['arquitectura']} as teacher",
                     "outputs/teacher.json")

## Resumen

In [ ]:
comparacion